In [ ]:
import pandas as pd
import os
from pinecone import Pinecone , ServerlessSpec
from dotenv import load_dotenv , find_dotenv

In [ ]:
files = pd.read_csv("description.csv" , encoding = "ANSI")

In [ ]:
def create_course_description(row):
    return f'''The course name is {row["course_name"]} the slug is{row["course_row"]},
            the technology is {row["course_technology"]} and the course topic is {row["course_topic"]}'''
    

In [ ]:
pd.set_option("display.max_rows" , 106)
files['course_description_new'] = files.apply(create_course_description , axis = 1)
print(files["course_description_new"])

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
load_dotenv(find_dotenv() , override = True)

In [ ]:
pc = Pinecone(api_key = os.environ.get("PINECONE_API_KEY") , environment = os.environment.get('PINECONE_ENV))

In [ ]:
index_name = "my_index"
dimension = 384
metric = "cosine"

In [ ]:
if index_name in [index.name for index in pc.list_indexes()]:
    pc.delete_index(index_name)
    print(f"{index_name} successfully deleted.")
else:
    print(f"{index_name} not in index_list")

In [ ]:
pc.create_index(
    name = index_name,
    dimension = dimension,
    metric = metric,
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
        
        
    )
)

In [ ]:
index = pc.Index(name = index_name)

# embedding the data

In [ ]:
model = SentenceTransformers("all-MiniLM-L6-v2")

In [ ]:
def create_embedding(rows):
    combined_text = " ".join([str(row[field]) for field in ['course_description' , 'course_description_new' , 'course_description'])
    embedding = model.encode(combined_text , show_progress_bar = False)
    return embedding
                             
                              
                                                            

In [ ]:
files["embedding"] = files.apply(create_embeddings , axis = 1)

In [ ]:
vectors_to_upsert = [(str(row["course_name"]) , row["embedding"].tolist()) for _, row in files.iterrows()]
index.upsert(vectors = vectors_to_upsert)

print("data upserted to pinecone index")

# semantic search

In [ ]:
query = "clustering"
query_embadding = model.encode(query , show_progress_bar = False).tolist()

In [ ]:
query_results = index.query(
    vector = [query_embadding],
    top_k = 12,
    include_values = True
)

In [ ]:
quary_results

In [ ]:
score_thresold = 0.3
for match in quary_results["matches"]:
    print(f"Marched item ID: {match['id']} , score: {match['score']}")
    